# NB-401: Quantization Techniques

Learn and apply model quantization for LLMs.

## Learning Objectives
- Dynamic quantization
- Post-training quantization
- Quantization-aware training
- Compare model sizes and performance

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.quantization as quantization

print("Libraries imported")

## 1. Create Sample Model

In [ ]:
# TODO: Create a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleModel()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Dynamic Quantization

In [ ]:
# TODO: Apply dynamic quantization
import torch.ao.quantization as quantization

# Configure quantization
model.qconfig = quantization.get_default_qconfig('fbgemm')

# Prepare model
model_prepared = quantization.prepare(model, inplace=False)

# Calibrate with sample data
dummy_input = torch.randn(1, 784)
for _ in range(100):
    _ = model_prepared(dummy_input)

# Convert to quantized model
model_quantized = quantization.convert(model_prepared)

print(f"Original size: {sum(p.numel() for p in model.parameters()):,}")
print(f"Quantized size: {sum(p.numel() for p in model_quantized.parameters()):,}")

## 3. Post-Training Quantization with Transformers

In [ ]:
# TODO: Load and quantize a transformer
from transformers import BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization
# model = AutoModelForCausalLM.from_pretrained(
#     "gpt2",
#     quantization_config=bnb_config,
#     device_map="auto"
# )

print("4-bit quantization configured")

## 4. Compare Model Performance

In [ ]:
import time

# TODO: Benchmark models
def benchmark_model(model, inputs, n_runs=100):
    model.eval()
    times = []

    with torch.no_grad():
        for _ in range(n_runs):
            start = time.time()
            _ = model(inputs)
            times.append(time.time() - start)

    return sum(times) / len(times)

# Benchmark
dummy_input = torch.randn(1, 784)

# original_time = benchmark_model(model, dummy_input)
# quantized_time = benchmark_model(model_quantized, dummy_input)

# print(f"Original: {original_time*1000:.2f}ms")
# print(f"Quantized: {quantized_time*1000:.2f}ms")
# print(f"Speedup: {original_time/quantized_time:.2f}x")

print("Benchmark function defined")

## Exercise

1. Apply different quantization types (int8, fp16)
2. Measure accuracy drop from quantization
3. Compare memory usage